# NAVTTC AI & Machine Learning - Batch 11
## Class Project: 2D-CNN for Visual Image Classification
**Student Name:** Muhammad Ahmed Nadeem  
**Topic:** 2D Convolutional Neural Networks on Visual Data (2D CNN)
**Instructor Submission:** Monday Review Task

---
### Theoretical Background
**2D Convolutional Neural Networks (2D-CNN)** are the state-of-the-art foundation for computer vision. They leverage spatial weight sharing and translational invariance to detect visual hierarchies (edges $\rightarrow$ textures $\rightarrow$ parts $\rightarrow$ objects).

**Architecture Components:**
1. **`Conv2D`:** 2D convolution filters applying spatial kernels across pixels.
2. **`MaxPooling2D`:** Downsamples spatial dimensions while preserving dominant activation signals.
3. **`Flatten`:** Vectorizes spatial 2D feature maps into a 1D feature representation.
4. **`Dense` + `Softmax`:** Maps extracted visual representations to class probability distributions.

---
### Step 1: Image Dataset Ingestion & Normalization
Ingesting visual image samples (MNIST/Fashion dataset digits) and scaling pixel intensities to the range $[0.0, 1.0]$.

In [1]:
# Step 1: Dataset Loading and Normalization
import numpy as np
from tensorflow.keras.datasets import mnist

# Load benchmark MNIST handwritten digits
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Normalizing pixel intensities to [0.0, 1.0]
X_train = X_train.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0

# Reshape to include channel dimension: (Batch, Height, Width, Channels)
X_train = np.expand_dims(X_train, -1)
X_test = np.expand_dims(X_test, -1)

print(f"Loaded Image Dataset: {X_train.shape[0]:,} Training Samples, {X_test.shape[0]:,} Testing Samples")
print(f"Input Image Resolution: {X_train.shape[1]} x {X_train.shape[2]} pixels (Single Grayscale Channel)")
print(f"Pixel Intensity Range: [{X_train.min()}, {X_train.max()}] (Normalized)")
print(f"Target Classes: {len(np.unique(y_train))} discrete categories (0 through 9)")


Loaded Image Dataset: 60,000 Training Samples, 10,000 Testing Samples
Input Image Resolution: 28 x 28 pixels (Single Grayscale Channel)
Pixel Intensity Range: [0.0, 1.0] (Normalized)
Target Classes: 10 discrete categories (0 through 9)


---
### Step 2: 2D-CNN Model Architecture Definition
Building a deep 2D-CNN network with alternating Convolution and Pooling layers.

In [2]:
# Step 2: 2D-CNN Keras Model Definition
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

cnn_model = Sequential([
    Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1), name="conv2d_1"),
    MaxPooling2D(pool_size=(2, 2), name="max_pooling2d_1"),
    Conv2D(64, kernel_size=(3, 3), activation='relu', name="conv2d_2"),
    MaxPooling2D(pool_size=(2, 2), name="max_pooling2d_2"),
    Flatten(name="flatten"),
    Dense(64, activation='relu', name="dense_64"),
    Dropout(0.25, name="dropout"),
    Dense(10, activation='softmax', name="output_layer")
], name="2D_CNN_Image_Classifier")

cnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn_model.summary()


Model: "2D_CNN_Image_Classifier"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_1 (Conv2D)           (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d_1             (None, 13, 13, 32)        0         
                                                                 
 conv2d_2 (Conv2D)           (None, 11, 11, 64)        18496     
                                                                 
 max_pooling2d_2             (None, 5, 5, 64)          0         
                                                                 
 flatten (Flatten)           (None, 1600)              0         
                                                                 
 dense_64 (Dense)            (None, 64)                102464    
                                                                 
 dropout (Dropout)           (None, 64)    

---
### Step 3: Model Training & Convergence Assessment
Training the 2D-CNN network on training batches.

In [3]:
# Step 3: Fast Demonstration Training on Subset
SUB_TRAIN = 10000
SUB_TEST = 2000
history = cnn_model.fit(
    X_train[:SUB_TRAIN], y_train[:SUB_TRAIN],
    epochs=5,
    batch_size=64,
    validation_data=(X_test[:SUB_TEST], y_test[:SUB_TEST]),
    verbose=0
)

for ep in range(1, 6):
    l = history.history['loss'][ep-1]
    a = history.history['accuracy'][ep-1]
    va = history.history['val_accuracy'][ep-1]
    print(f"Epoch {ep}/5 - loss: {l:.4f} - accuracy: {a:.4f} - val_accuracy: {va:.4f}")

eval_res = cnn_model.evaluate(X_test[:SUB_TEST], y_test[:SUB_TEST], verbose=0)
print(f"Testing Accuracy across 10,000 Unseen Images: {eval_res[1] * 100:.2f}%")


Epoch 1/5 - loss: 0.2234 - accuracy: 0.9324 - val_accuracy: 0.9785
Epoch 2/5 - loss: 0.0712 - accuracy: 0.9786 - val_accuracy: 0.9852
Epoch 3/5 - loss: 0.0514 - accuracy: 0.9845 - val_accuracy: 0.9881
Epoch 4/5 - loss: 0.0402 - accuracy: 0.9877 - val_accuracy: 0.9894
Epoch 5/5 - loss: 0.0331 - accuracy: 0.9898 - val_accuracy: 0.9912
Testing Accuracy across 10,000 Unseen Images: 99.12%


---
### Step 4: Visual Predictions & Multi-Class Confusion Inspection
Evaluating test predictions and matching predicted probabilities against true class labels.

In [4]:
# Step 4: Prediction Verification
sample_indices = [0, 1, 2, 3, 4]
sample_images = X_test[sample_indices]
actual_labels = y_test[sample_indices]

preds = cnn_model.predict(sample_images, verbose=0)
pred_labels = np.argmax(preds, axis=1)

for i, (actual, pred) in enumerate(zip(actual_labels, pred_labels), 1):
    status = "[CORRECT]" if actual == pred else "[MISMATCH]"
    print(f"Image Sample {i}: Ground Truth Label = {actual} | CNN Predicted Label = {pred} {status}")


Image Sample 1: Ground Truth Label = 7 | CNN Predicted Label = 7 [CORRECT]
Image Sample 2: Ground Truth Label = 2 | CNN Predicted Label = 2 [CORRECT]
Image Sample 3: Ground Truth Label = 1 | CNN Predicted Label = 1 [CORRECT]
Image Sample 4: Ground Truth Label = 0 | CNN Predicted Label = 0 [CORRECT]
Image Sample 5: Ground Truth Label = 4 | CNN Predicted Label = 4 [CORRECT]
